## 1. Setup & Load Dataset

In [ ]:
# Core imports
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from IPython.display import Video, display
from tqdm.auto import tqdm

# Check PyTorch availability
import torch
print(f"✅ PyTorch {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

# Verify dataset exists
dataset_path = Path("RouletteVision-Dataset")
assert dataset_path.exists(), "Dataset not found! Make sure it's cloned."
print(f"\n✅ Dataset found at: {dataset_path}")

## 2. Explore Dataset Structure

In [ ]:
# Count videos in each set
sets = ["SET 1", "SET 2", "SET 3", "SET 4"]
video_counts = {}

for set_name in sets:
    input_path = dataset_path / "Input-Output Videos" / set_name
    input_videos = list(input_path.glob("*_INPUT_*.mp4"))
    output_videos = list(input_path.glob("*_OUTPUT_*.mp4"))
    video_counts[set_name] = {"input": len(input_videos), "output": len(output_videos)}
    
print("Dataset Structure:")
print("-" * 50)
for set_name, counts in video_counts.items():
    print(f"{set_name}: {counts['input']} input videos, {counts['output']} output videos")
print("-" * 50)
total_videos = sum(c['input'] for c in video_counts.values())
print(f"Total: {total_videos} video pairs")

## 3. Inspect Sample Videos

In [ ]:
# Helper function to get video properties
def get_video_info(video_path):
    """Extract video metadata"""
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = frame_count / fps if fps > 0 else 0
    cap.release()
    return {
        "fps": fps,
        "frames": frame_count,
        "resolution": (width, height),
        "duration": duration
    }

# Load sample video pair
sample_input = dataset_path / "Input-Output Videos/SET 1/S1_INPUT_1.mp4"
sample_output = dataset_path / "Input-Output Videos/SET 1/S1_OUTPUT_1.mp4"

input_info = get_video_info(sample_input)
output_info = get_video_info(sample_output)

print("📹 INPUT VIDEO (ball spinning):")
print(f"   Resolution: {input_info['resolution'][0]}x{input_info['resolution'][1]}")
print(f"   Duration: {input_info['duration']:.2f}s")
print(f"   FPS: {input_info['fps']:.1f}")
print(f"   Frames: {input_info['frames']}")

print("\n🎯 OUTPUT VIDEO (ball landing):")
print(f"   Resolution: {output_info['resolution'][0]}x{output_info['resolution'][1]}")
print(f"   Duration: {output_info['duration']:.2f}s")
print(f"   FPS: {output_info['fps']:.1f}")
print(f"   Frames: {output_info['frames']}")

In [ ]:
# Display sample videos
print("📹 Sample INPUT video (ball spinning around wheel):")
display(Video(str(sample_input), width=500, embed=True))

print("\n🎯 Sample OUTPUT video (ball landing in number):")
display(Video(str(sample_output), width=500, embed=True))

## 4. Visualize Output Video Frames

Let's look at frames from the output video to see if we can identify the winning number.

In [ ]:
def extract_frames(video_path, num_frames=6):
    """Extract evenly spaced frames from video"""
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_indices = np.linspace(0, total_frames-1, num_frames, dtype=int)
    
    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames

# Extract and display frames from output video
output_frames = extract_frames(sample_output, num_frames=6)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()
for i, frame in enumerate(output_frames):
    axes[i].imshow(frame)
    axes[i].set_title(f"Frame {i+1}/6", fontsize=12)
    axes[i].axis('off')
plt.suptitle("OUTPUT VIDEO - Ball Landing Sequence", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n🔍 Can you see the winning number displayed in any frame?")
print("This will determine our label extraction strategy.")

## 5. Build Dataset Index

Create a comprehensive index of all video pairs with metadata.

In [ ]:
# Build dataset index
dataset_records = []

print("Building dataset index...")
for set_name in tqdm(sets, desc="Processing sets"):
    input_dir = dataset_path / "Input-Output Videos" / set_name
    input_videos = sorted(input_dir.glob("*_INPUT_*.mp4"))
    
    for input_video in input_videos:
        # Find corresponding output video
        video_id = input_video.stem.split("_INPUT_")[1]
        set_prefix = input_video.stem.split("_INPUT_")[0]
        output_video = input_dir / f"{set_prefix}_OUTPUT_{video_id}.mp4"
        
        if output_video.exists():
            try:
                input_info = get_video_info(input_video)
                output_info = get_video_info(output_video)
                
                dataset_records.append({
                    "set": set_name,
                    "video_id": int(video_id),
                    "input_path": str(input_video.relative_to(dataset_path)),
                    "output_path": str(output_video.relative_to(dataset_path)),
                    "input_duration": input_info["duration"],
                    "input_frames": input_info["frames"],
                    "output_duration": output_info["duration"],
                    "output_frames": output_info["frames"],
                    "label": None  # To be filled with winning number (0-36)
                })
            except Exception as e:
                print(f"Error processing {input_video.name}: {e}")

# Create DataFrame
df = pd.DataFrame(dataset_records)
df = df.sort_values(['set', 'video_id']).reset_index(drop=True)

print(f"\n✅ Dataset index created: {len(df)} video pairs")
print(f"\nBreakdown by set:")
print(df.groupby('set').size())
print(f"\nSample entries:")
display(df.head())

## 6. ⚠️ CRITICAL: Label Extraction Strategy

**We need to extract the winning number (0-36) for each video.**

### Options:
1. **Check GitHub repo** - Look for existing labels or extraction code
2. **OCR** - Extract numbers from video frames (if displayed)
3. **Computer Vision** - Track ball position and map to wheel
4. **Manual labeling** - Use GUI to speed up manual process

### Recommendation for 2-day timeline:
- **Option 1**: Check GitHub first (5 min)
- **Option 4**: Create quick labeling tool, label 300-500 videos (2-3 hours)
- This is sufficient for a strong demo!

## 7. Next Steps

### Immediate priorities:
1. ✅ Dataset explored and indexed
2. ⏳ **Solve labeling** (biggest blocker)
3. ⏳ Create PyTorch Dataset class
4. ⏳ Set up model architecture (ResNet3D + LSTM)
5. ⏳ Train overnight
6. ⏳ Build demo notebook tomorrow

### Save dataset index

In [ ]:
# Save dataset index to CSV
output_file = "dataset_index.csv"
df.to_csv(output_file, index=False)
print(f"✅ Dataset index saved to: {output_file}")
print(f"\nNext: Add 'label' column with winning numbers (0-36) for each video.")